In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


USE_MANUAL_5050_TEST = False   # True = test 50/50, False = stratified kfold

# Caricamento dati

In [8]:
FILE_PATH = Path("/Users/francesco/Tesi/BC-ML4/dataset/cleaned")
df = pd.read_csv(FILE_PATH / "ambl_lesions.csv")

df["PR_class"] = (pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1).astype(int)
df = df.dropna(subset=["PR_class"])

print("Totale campioni:", len(df))
print("Distribuzione PR_class:")
print(df["PR_class"].value_counts())


Totale campioni: 82
Distribuzione PR_class:
PR_class
0    46
1    36
Name: count, dtype: int64


# Target

In [9]:
drop_cols = [
    "Patient ID","lesion idx","tumor/benign",
    "GRADE","isTN","Breast",
    "ER [SII]","PR [SII]","HER2 [SII]",
    "PR_class"
]

groups = df["Patient ID"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["PR_class"]

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean())


# Split 50/50

In [10]:
def create_manual_5050_splits(X, y, n_splits=5, random_state=42):
    np.random.seed(random_state)

    idx0 = y[y == 0].index.to_numpy()
    idx1 = y[y == 1].index.to_numpy()

    np.random.shuffle(idx0)
    np.random.shuffle(idx1)

    n_min = min(len(idx0), len(idx1))
    n_test = n_min // n_splits

    splits = []
    for k in range(n_splits):
        test0 = idx0[k*n_test:(k+1)*n_test]
        test1 = idx1[k*n_test:(k+1)*n_test]

        test_idx = np.concatenate([test0, test1])
        train_idx = np.setdiff1d(np.arange(len(y)), test_idx)

        splits.append((train_idx, test_idx))

    return splits


# Tutto il resto

In [11]:
if USE_MANUAL_5050_TEST:
    print("\n>>> USO TEST SET MANUALE 50/50")
    cv_splits = create_manual_5050_splits(X, y, n_splits=5, random_state=42)
else:
    print("\n>>> USO STRATIFIED K-FOLD")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_splits = list(skf.split(X, y))


acc_scores, bal_scores, f1_scores, auc_scores = [], [], [], []
results_rows = []

print("\n==============================")
print("TARGET: PR (AMBL)")
print("==============================")

for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_test  = y.iloc[test_idx]

    print(f"\n{'='*40}")
    print(f"FOLD {fold}")
    print(f"{'='*40}")
    print(f"TRAIN - Classe 0: {(y_train==0).sum()} | Classe 1: {(y_train==1).sum()}")
    print(f"TEST  - Classe 0: {(y_test==0).sum()} | Classe 1: {(y_test==1).sum()}")

    # ====================================================
    # SCALE_POS_WEIGHT (SOLO TRAIN)
    # ====================================================

    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

    print(f"scale_pos_weight: {scale_pos_weight:.4f}")

    # ====================================================
    # MODELLO
    # ====================================================

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=1,
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        n_jobs=1
    )

    model.fit(X_train, y_train)

    # ====================================================
    # FEATURE IMPORTANCE
    # ====================================================

    importances = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    print("\nTop 20 Feature Importance:")
    display(fi_df.head(20))

    # fi_df.to_csv(f"feature_importance_fold_{fold}.csv", index=False)

    # ====================================================
    # PREDICTIONS
    # ====================================================

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    # ====================================================
    # METRICHE
    # ====================================================

    acc = accuracy_score(y_test, y_pred)
    bal = balanced_accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    acc_scores.append(acc)
    bal_scores.append(bal)
    f1_scores.append(f1)
    auc_scores.append(auc)

    print(f"\nAccuracy: {acc:.4f}")
    print(f"Balanced Accuracy: {bal:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")



>>> USO STRATIFIED K-FOLD

TARGET: PR (AMBL)

FOLD 1
TRAIN - Classe 0: 36 | Classe 1: 29
TEST  - Classe 0: 10 | Classe 1: 7
scale_pos_weight: 1.2414

Top 20 Feature Importance:


,Feature,Importance
84,original_glszm_GrayLevelNonUniformity,0.049840
95,original_glszm_SmallAreaHighGrayLevelEmphasis,0.037737
10,original_shape_Sphericity,0.035737
44,original_glcm_Imc1,0.032146
19,original_firstorder_Maximum,0.032140
5,original_shape_Maximum2DDiameterRow,0.030942
14,original_firstorder_10Percentile,0.030488
100,original_ngtdm_Busyness,0.029769
88,original_glszm_LargeAreaEmphasis,0.028921
86,original_glszm_GrayLevelVariance,0.026981



Accuracy: 0.7059
Balanced Accuracy: 0.6857
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        10
           1       0.67      0.57      0.62         7

    accuracy                           0.71        17
   macro avg       0.70      0.69      0.69        17
weighted avg       0.70      0.71      0.70        17

Confusion Matrix:
[[8 2]
 [3 4]]
TN=8 | FP=2 | FN=3 | TP=4

FOLD 2
TRAIN - Classe 0: 37 | Classe 1: 28
TEST  - Classe 0: 9 | Classe 1: 8
scale_pos_weight: 1.3214

Top 20 Feature Importance:


,Feature,Importance
104,original_ngtdm_Strength,0.058379
50,original_glcm_MCC,0.035928
96,original_glszm_SmallAreaLowGrayLevelEmphasis,0.035268
89,original_glszm_LargeAreaHighGrayLevelEmphasis,0.033481
53,original_glcm_SumEntropy,0.029111
76,original_glrlm_RunEntropy,0.029016
2,original_shape_LeastAxisLength,0.028777
80,original_glrlm_RunVariance,0.028219
39,original_glcm_DifferenceVariance,0.028085
65,original_gldm_SmallDependenceEmphasis,0.026899



Accuracy: 0.7059
Balanced Accuracy: 0.6875
              precision    recall  f1-score   support

           0       0.64      1.00      0.78         9
           1       1.00      0.38      0.55         8

    accuracy                           0.71        17
   macro avg       0.82      0.69      0.66        17
weighted avg       0.81      0.71      0.67        17

Confusion Matrix:
[[9 0]
 [5 3]]
TN=9 | FP=0 | FN=5 | TP=3

FOLD 3
TRAIN - Classe 0: 37 | Classe 1: 29
TEST  - Classe 0: 9 | Classe 1: 7
scale_pos_weight: 1.2759

Top 20 Feature Importance:


...


Accuracy: 0.5625
Balanced Accuracy: 0.5476
              precision    recall  f1-score   support

           0       0.60      0.67      0.63         9
           1       0.50      0.43      0.46         7

    accuracy                           0.56        16
   macro avg       0.55      0.55      0.55        16
weighted avg       0.56      0.56      0.56        16

Confusion Matrix:
[[6 3]
 [4 3]]
TN=6 | FP=3 | FN=4 | TP=3

FOLD 4
TRAIN - Classe 0: 37 | Classe 1: 29
TEST  - Classe 0: 9 | Classe 1: 7
scale_pos_weight: 1.2759

Top 20 Feature Importance:


,Feature,Importance
100,original_ngtdm_Busyness,0.052833
96,original_glszm_SmallAreaLowGrayLevelEmphasis,0.039047
54,original_glcm_SumSquares,0.035763
103,original_ngtdm_Contrast,0.035343
61,original_gldm_LargeDependenceEmphasis,0.033337
6,original_shape_Maximum2DDiameterSlice,0.033321
91,original_glszm_LowGrayLevelZoneEmphasis,0.032588
10,original_shape_Sphericity,0.029359
58,original_gldm_DependenceVariance,0.028534
92,original_glszm_SizeZoneNonUniformity,0.028167



Accuracy: 0.5000
Balanced Accuracy: 0.5079
              precision    recall  f1-score   support

           0       0.57      0.44      0.50         9
           1       0.44      0.57      0.50         7

    accuracy                           0.50        16
   macro avg       0.51      0.51      0.50        16
weighted avg       0.52      0.50      0.50        16

Confusion Matrix:
[[4 5]
 [3 4]]
TN=4 | FP=5 | FN=3 | TP=4

FOLD 5
TRAIN - Classe 0: 37 | Classe 1: 29
TEST  - Classe 0: 9 | Classe 1: 7
scale_pos_weight: 1.2759

Top 20 Feature Importance:


,Feature,Importance
50,original_glcm_MCC,0.047490
61,original_gldm_LargeDependenceEmphasis,0.034177
10,original_shape_Sphericity,0.032083
89,original_glszm_LargeAreaHighGrayLevelEmphasis,0.032032
102,original_ngtdm_Complexity,0.029403
65,original_gldm_SmallDependenceEmphasis,0.029272
100,original_ngtdm_Busyness,0.027459
45,original_glcm_Imc2,0.027138
57,original_gldm_DependenceNonUniformityNormalized,0.027030
44,original_glcm_Imc1,0.026680



Accuracy: 0.6875
Balanced Accuracy: 0.6905
              precision    recall  f1-score   support

           0       0.75      0.67      0.71         9
           1       0.62      0.71      0.67         7

    accuracy                           0.69        16
   macro avg       0.69      0.69      0.69        16
weighted avg       0.70      0.69      0.69        16

Confusion Matrix:
[[6 3]
 [2 5]]
TN=6 | FP=3 | FN=2 | TP=5


# CSV


In [12]:
row = {
    "Dataset": "AMBL",
    "Target": "PR",
    "Accuracy": f"{np.mean(acc_scores):.3f} ± {np.std(acc_scores):.3f}",
    "Balanced Accuracy": f"{np.mean(bal_scores):.3f} ± {np.std(bal_scores):.3f}",
    "F1-score": f"{np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}",
    "ROC-AUC": f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
}

results_df = pd.DataFrame([row])
display(results_df)



,Dataset,Target,Accuracy,Balanced Accuracy,F1-score,ROC-AUC
0,AMBL,PR,0.632 ± 0.085,0.624 ± 0.079,0.558 ± 0.075,0.625 ± 0.119
